# Gemma4GR — E4B Greek STT Fine-tuning (Colab)
**Requirements:** L4 (40 GB) or A100 (80 GB) GPU

**Before running:**
1. Runtime → Change runtime type → GPU → L4 or A100
2. Add `HF_TOKEN` in Colab Secrets (🔑 icon on left sidebar)
3. Upload your `data/` folder to Google Drive under `Gemma4GR/`

In [ ]:
# Cell 1 — Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.get_device_properties(0).total_memory / 1e9 >= 15, \
    'Need L4 (40 GB) or A100 (80 GB) for E4B. Switch runtime type.'

In [ ]:
# Cell 2 — Install dependencies
!pip install unsloth[colab-new] trl datasets soundfile jiwer -q
!pip install torchcodec -q

In [ ]:
# Cell 3 — Mount Drive and clone repo
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

# Clone your repo (update URL)
REPO_URL = 'https://github.com/Efs-O/Gemma4GR'
!git clone {REPO_URL} /content/Gemma4GR 2>/dev/null || (cd /content/Gemma4GR && git pull)
%cd /content/Gemma4GR

# Set HF token from Colab Secrets
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF_TOKEN set ✓')

In [ ]:
# Cell 4 — Link data from Google Drive
import os
from pathlib import Path

DRIVE_DATA = '/content/drive/MyDrive/Gemma4GR/data'
LOCAL_DATA = '/content/Gemma4GR/data'

Path(LOCAL_DATA).mkdir(exist_ok=True)

# Symlink the audio folders (no copy, instant)
for folder in ['resampled_audio', 'transcripts']:
    src = f'{DRIVE_DATA}/{folder}'
    dst = f'{LOCAL_DATA}/{folder}'
    if Path(src).exists() and not Path(dst).exists():
        os.symlink(src, dst)
        print(f'  Linked: {folder}')
    elif Path(dst).exists():
        print(f'  Already present: {folder}')
    else:
        print(f'  [WARN] Not found in Drive: {src}')

# Copy JSONL files if they exist in Drive
import shutil
for fname in ['train_stt.jsonl', 'train_stt_val.jsonl']:
    src = f'{DRIVE_DATA}/{fname}'
    dst = f'{LOCAL_DATA}/{fname}'
    if Path(src).exists():
        shutil.copy2(src, dst)
        print(f'  Copied: {fname}')
    else:
        print(f'  [INFO] {fname} not in Drive — will regenerate')

In [ ]:
# Cell 5 — Regenerate JSONL if needed (paths update to Colab paths)
import os
if not os.path.exists('/content/Gemma4GR/data/train_stt.jsonl'):
    print('Regenerating JSONL with Colab paths ...')
    !python training/prepare_stt_dataset.py
else:
    import json
    with open('/content/Gemma4GR/data/train_stt.jsonl') as f:
        count = sum(1 for _ in f)
    print(f'JSONL ready: {count} training examples')

In [ ]:
# Cell 6 — Train E4B
!python training/train_e4b_colab.py

In [ ]:
# Cell 7 — Save output back to Google Drive
import shutil
from pathlib import Path

src = '/content/Gemma4GR/output/e4b_greek_stt'
dst = '/content/drive/MyDrive/Gemma4GR/output/e4b_greek_stt'

if Path(src).exists():
    if Path(dst).exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Saved to Drive: {dst}')
else:
    print('[ERROR] No output found — training may have failed')

In [ ]:
# Cell 8 — Quick test inference
# Upload a Greek WAV file to /content/test.wav to test
import os
test_wav = '/content/test.wav'

if os.path.exists(test_wav):
    from unsloth import FastModel
    from unsloth.chat_templates import get_chat_template

    model, processor = FastModel.from_pretrained(
        model_name='/content/Gemma4GR/output/e4b_greek_stt/lora_adapter',
        load_in_4bit=True,
        max_seq_length=4096,
    )

    messages = [{
        'role': 'user',
        'content': [
            {'type': 'audio', 'audio': test_wav},
            {'type': 'text',  'text': 'Transcribe the following speech segment in Greek into Greek text.'}
        ]
    }]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_tensors='pt', add_generation_prompt=True
    ).to('cuda')

    with __import__('torch').no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)

    result = processor.tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Transcription: {result}')
else:
    print('Upload a Greek WAV file as /content/test.wav to test inference')